# Chord moves — native re-run (no PowerShell)

Operator: "why are we using powershell cant we run it via mag tools".

The right surface is `mag.headless.run_headless` — the canonical headless spawner
(CREATE_NO_WINDOW + the real interpreter + reliable captured stdout), the SAME runner
the k8s swarm_worker uses. This notebook runs through the Python kernel (a mag-native
execution surface) instead of a PowerShell terminal.

Cell 1 dogfoods the new `recurrence_gate` (this session's PowerShell default WAS a
`wrong_tool` instance). Cell 2 re-runs the full RIB verification via `headless.run_headless`.


In [ ]:
# Cell 1 — dogfood the gate: record the wrong_tool recurrence + show the guard opening
import sys, json
from pathlib import Path
ROOT = Path.cwd() if (Path.cwd() / "config.py").is_file() else Path(r"c:\Users\foste\Documents\projects\local_sovereign_agent")
sys.path.insert(0, str(ROOT))

from mag import recurrence_gate

# honest dogfood: this session defaulted to a PowerShell terminal for framework work = wrong_tool
rec = recurrence_gate.record("wrong_tool",
    "RIB verification ran through run_in_terminal (PowerShell) instead of mag.headless — "
    "the documented wrong tool for framework work; corrected to the headless native runner.")
print("recorded:", rec["ok"])

# confirm the gate will now flag wrong_tool at threshold 1 within the window
g = recurrence_gate.check(window_hours=24, threshold=1)
print("guards_opened:", json.dumps(g.get("guards_opened", {}), indent=2))
print("note:", g.get("note"))


In [ ]:
# Cell 2 — re-run the RIB verification through the CANONICAL headless runner (no window, no terminal)
from mag import headless as hl
from mag.headless import _resolve_python

py = _resolve_python()
r = hl.run_headless(
    [py, "_verify_chord_moves.py"],
    capture_output=True, text=True, encoding="utf-8", errors="replace", timeout=180,
)
print("rc:", r.returncode)
print("---- stdout ----")
print(r.stdout[-4000:] if r.stdout else "(no stdout)")
if r.stderr:
    print("---- stderr ----")
    print(r.stderr[-1500:])


In [ ]:
# Cell 3 — billing gate: confirm the '{}' from the PowerShell run was the SEEDING path, not a bug.
# gate() early-returns (no metrics) when the ledger has < min_rows; metrics appear once fed.
from mag import billing_feed
import json

g = billing_feed.gate()
print("ok:", g.get("ok"), "| gate_triggered:", g.get("gate_triggered"), "| n_rows:", g.get("n_rows"))
print("has_metrics_key:", "metrics" in g)
print("note:", g.get("note"))

# show the contract
st = billing_feed.status()
print("ledger:", st.get("ledger"), "| n_rows:", st.get("n_rows"))
